In [ ]:
# Cell 1: Setup Environment and Download Dataset
import os
import sys
import urllib.request
import pandas as pd
from pathlib import Path

REPO_NAME = ""
# 1. Setup Environment
REPO_URL = "https://github.com/b01767769/retail-model-drift.git"
if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}
    print(f"Successfully cloned {REPO_NAME}")
else:
    print(f"Repository {REPO_NAME} already exists. Pulling latest...")
    !cd {REPO_NAME} && git pull
REPO_NAME = REPO_URL.split("/")[-1].replace(".git", "")


project_root = Path(os.getcwd()).parent
target_dir = REPO_NAME if os.path.exists(REPO_NAME) else project_root

os.chdir(target_dir)

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

print(f"Current Working Directory: {os.getcwd()}")

# 2. Download and Format Dataset
Path("data/raw").mkdir(parents=True, exist_ok=True)
Path("data/processed").mkdir(parents=True, exist_ok=True)
Path("artifacts").mkdir(parents=True, exist_ok=True)

csv_path = Path("data/raw/online_retail_II.csv")
excel_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00502/online_retail_II.xlsx"
excel_path = Path("data/raw/online_retail_II.xlsx")

if not csv_path.exists():
    print("Downloading dataset from UCI Machine Learning Repository...")
    urllib.request.urlretrieve(excel_url, excel_path)
    print("Converting Excel to CSV...")
    df_excel = pd.read_excel(excel_path, sheet_name="Year 2010-2011") 
    df_excel.to_csv(csv_path, index=False)
    excel_path.unlink()
    print("Dataset ready at:", csv_path)
else:
    print("Dataset already exists at:", csv_path)
%pip install -r requirements.txt

In [ ]:
# Cell 2: Master Configuration & Imports
import mlflow
import pandas as pd
from pathlib import Path

# Import the refactored modules
from src.data_ingest import load_data
from src.preprocess import clean_and_slice_data
from src.features import RFMFeatureEngineer, assign_targets
from src.train import train_baseline_rf
from src.drift import extract_quantile_bins, compute_psi_report, generate_drift_artifacts, check_drift_trigger
from src.retrain import execute_challenger_retraining
from src.evaluate import evaluate_model_bootstrap, compare_models, compile_evaluation_report
from src.mlflow_utils import generate_pipeline_manifest, log_run_standard, register_promoted_champion

mlflow.set_tracking_uri("sqlite:///mlflow.db")
EXPERIMENT_NAME = "retail_mlops_champion_challenger"

# MASTER PIPELINE CONFIGURATION 
# Matches the strict MLflow REQUIRED_PARAMS schema
PIPELINE_CONFIG = {
    # Engineering Params
    "winsorize_percentile": 99.5,
    "log_transform_monetary": True,
    "apply_scaling": True,
    "target_percentile": 75.0,
    
    # Model Params
    "model_type": "RandomForest",
    "n_estimators": 200,
    "max_depth": 10,
    "min_samples_leaf": 5,
    "random_seed": 42,
    
    # Drift & Retrain Params
    "psi_bins": 10,
    "psi_min_count": 50, # satisfies psi_min_count schema requirement
    "psi_min_bin_count": 5,
    "psi_threshold": 0.25,
    "auc_tolerance": 0.03,
    "promotion_min_improvement": 0.02,
    "retrain_strategy": "cumulative",
    "retrain_sliding_window_size": 2,
    
    # Tracking
    "train_slices": "1" # Will update dynamically during loop
}

In [ ]:
# Cell 3: Data Pipeline Execution
raw_df = load_data("data/raw/online_retail_II.csv")
slices_dict = clean_and_slice_data(raw_df, num_slices=6)

rfm_slices = {}
slice_diagnostics = {}

# 1. Initialize Stateful Feature Engineer
engineer = RFMFeatureEngineer(config=PIPELINE_CONFIG)

# 2. Extract Features safely across slices
for slice_num, slice_df in slices_dict.items():
    if slice_num == 1:
        # Lock in parameters (caps, scaler) on baseline
        rfm_slices[slice_num], slice_diagnostics[slice_num] = engineer.fit_transform(slice_df)
    else:
        # Apply parameters securely to future data
        rfm_slices[slice_num], slice_diagnostics[slice_num] = engineer.transform(slice_df)

# 3. Assign Targets based on future behavior
for i in range(1, 6): 
    rfm_slices[i] = assign_targets(
        current_rfm=rfm_slices[i], 
        next_rfm=rfm_slices[i+1], 
        config=PIPELINE_CONFIG
    )
    print(f"Targets assigned for Slice {i}")

In [ ]:
# Cell 4: Train Initial Champion (Slice 1)
baseline_data = rfm_slices[1]

champion_model, train_metrics = train_baseline_rf(baseline_data, config=PIPELINE_CONFIG)

baseline_bins = extract_quantile_bins(
    baseline_data, 
    feature_cols=['recency_scaled', 'frequency_scaled', 'monetary_log_scaled'], 
    num_bins=PIPELINE_CONFIG["psi_bins"]
)

champion_eval_metrics = evaluate_model_bootstrap(champion_model, baseline_data, random_seed=PIPELINE_CONFIG['random_seed'])

manifest_path = generate_pipeline_manifest(
    data_source_path="data/raw/online_retail_II.csv",
    slice_boundaries={"slice_1": "Baseline"},
    run_role="baseline_champion",
    slice_number=1
)

pd.DataFrame([{"Note": "Baseline - No PSI"}]).to_csv("artifacts/psi_report.csv", index=False)
with open("artifacts/run_notes.txt", "w") as f: f.write("Initial Champion Model Deployment.")

run_artifacts = {
    "model": champion_model,
    "scaler": engineer.scaler,
    "feature_importance_plot": "artifacts/feature_importance.png",
    "psi_report_csv": "artifacts/psi_report.csv",
    "confusion_matrix": "artifacts/confusion_matrix.png",
    "run_notes": "artifacts/run_notes.txt"
}

run_metrics = {
    **champion_eval_metrics, 
    "val_loss": train_metrics["val_loss"], 
    "psi_recency": 0.0, 
    "psi_frequency": 0.0, 
    "psi_monetary": 0.0
}

run_tags = {"run_role": "baseline_champion", "slice_number": 1, "drift_trigger": "no", "promotion": "yes"}

baseline_run_id = log_run_standard(PIPELINE_CONFIG, run_metrics, run_tags, run_artifacts, EXPERIMENT_NAME)

print(f"Baseline Champion trained. Validation AUC: {champion_eval_metrics['auc']:.4f}")

In [ ]:
# Cell 5: MLOps Production Loop (Slices 2 to 5)
historical_champion_auc = champion_eval_metrics['auc']

for slice_num in range(2, 6):
    print(f"\n{'='*40}\n--- Evaluating Slice {slice_num} ---\n{'='*40}")
    current_data = rfm_slices[slice_num]
    
    # 1. Distributional Monitoring (PSI)
    psi_report, psi_df = compute_psi_report(
        baseline_bins, current_data, 
        min_customers=PIPELINE_CONFIG["psi_min_count"], 
        min_bin_count=PIPELINE_CONFIG["psi_min_bin_count"]
    )
    drift_artifacts = generate_drift_artifacts(psi_report, psi_df)
    
    # FIX: Map scaled column names back to MLflow strict schema names
    if psi_report["is_reliable"]:
        psi_metrics = {
            "psi_recency": float(psi_report["features"].get("recency_scaled", 0.0)),
            "psi_frequency": float(psi_report["features"].get("frequency_scaled", 0.0)),
            "psi_monetary": float(psi_report["features"].get("monetary_log_scaled", 0.0))
        }
    else:
        # Failsafe if not reliable
        psi_metrics = {"psi_recency": 0.0, "psi_frequency": 0.0, "psi_monetary": 0.0}
    
    # 2. Evaluate current Champion
    champion_eval_metrics = evaluate_model_bootstrap(champion_model, current_data, random_seed=PIPELINE_CONFIG['random_seed'])
    auc_drop = historical_champion_auc - champion_eval_metrics['auc']
    print(f"Champion AUC: {champion_eval_metrics['auc']:.4f} (Drop vs Historical: {auc_drop:.4f})")
    
    # 3. Check Corroborated Trigger
    is_triggered, trigger_summary = check_drift_trigger(psi_report, auc_drop, PIPELINE_CONFIG)
    
    # Base Logging Setup
    run_metrics = {**champion_eval_metrics, "val_loss": champion_eval_metrics['brier_score'], **psi_metrics} # use brier as proxy if not training
    run_artifacts.update({"psi_report_csv": drift_artifacts["psi_report_csv"]})
    run_tags = {"slice_number": slice_num, "drift_trigger": "yes" if is_triggered else "no", "promotion": "no"}
    PIPELINE_CONFIG["train_slices"] = str(slice_num) # Update config param for tracking
    
    with open("artifacts/run_notes.txt", "w") as f: f.write(trigger_summary)
    
    if is_triggered:
        print(f">> Corroborated Drift Detected! {trigger_summary}\n>> Training Challenger...")
        
        # 4. Train Challenger
        challenger_model, chal_train_metrics = execute_challenger_retraining(
            slice_repository=rfm_slices, current_slice_id=slice_num, config=PIPELINE_CONFIG
        )
        challenger_eval_metrics = evaluate_model_bootstrap(challenger_model, current_data, random_seed=PIPELINE_CONFIG['random_seed'])
        
        # Update metrics for challenger logging
        run_metrics = {**challenger_eval_metrics, "val_loss": chal_train_metrics["val_loss"], **psi_metrics}
        PIPELINE_CONFIG["train_slices"] = f"1 to {slice_num-1}"
        run_artifacts["model"] = challenger_model
        
        # 5. Compare & Promote
        promote, comp_report = compare_models(
            champion_eval_metrics, challenger_eval_metrics, min_auc_improvement=PIPELINE_CONFIG["promotion_min_improvement"]
        )
        
        if promote:
            print(f">> Challenger Promoted! New AUC: {challenger_eval_metrics['auc']:.4f}")
            champion_model = challenger_model
            historical_champion_auc = challenger_eval_metrics['auc']
            
            run_tags.update({"run_role": "promoted_champion", "promotion": "yes"})
            run_id = log_run_standard(PIPELINE_CONFIG, run_metrics, run_tags, run_artifacts, EXPERIMENT_NAME)
            
            register_promoted_champion(
                run_id=run_id, model_registry_name="Retail_RFM_Predictor",
                changelog_entry=f"Retrained on historical data prior to slice {slice_num}.",
                psi_trigger_summary=trigger_summary,
                validation_results_summary=f"AUC improved by {comp_report['delta_auc']:.4f}"
            )
        else:
            print(f">> Challenger rejected. Did not meet promotion criteria. Delta AUC: {comp_report['delta_auc']:.4f}")
            run_tags.update({"run_role": "rejected_challenger"})
            run_id = log_run_standard(PIPELINE_CONFIG, run_metrics, run_tags, run_artifacts, EXPERIMENT_NAME)
            
    else:
        print(">> System stable. No retraining required.")
        run_tags.update({"run_role": "champion_evaluation"})
        run_id = log_run_standard(PIPELINE_CONFIG, run_metrics, run_tags, run_artifacts, EXPERIMENT_NAME)
    
    # Generate the Per-Slice HTML Report
    compile_evaluation_report(slice_num, run_metrics, run_id, trigger_summary)

In [ ]:
# Cell 6: Backup Artifacts and Database
import shutil
from google.colab import files

print("Zipping MLflow database and artifacts folder...")
shutil.make_archive("mlops_artifacts", "zip", "artifacts")
#shutil.make_archive("mlruns_backup", "zip", "mlruns")

files.download("mlflow.db")
files.download("mlops_artifacts.zip")
#files.download("mlruns_backup.zip")
print("Downloads initiated. You have the DB, local artifacts, and mlruns backend.")

In [ ]:
# Cell 7: Generate Chapter 6 Markdown Reports
import mlflow
import pandas as pd
from IPython.display import display, Markdown

def generate_slice_report(slice_num: int, experiment_name: str) -> str:
    """Queries MLflow and formats the results into a comprehensive Audit specification."""
    
    exp = mlflow.get_experiment_by_name(experiment_name)
    if not exp:
        return f"Error: Experiment '{experiment_name}' not found."
        
    runs_df = mlflow.search_runs(experiment_ids=[exp.experiment_id])
    
    if runs_df.empty:
        return "No runs found. Did the pipeline execute?"

    # Filter for the specific slice
    # Note: MLflow logs parameters as strings
    slice_runs = runs_df[runs_df["tags.slice_number"] == str(slice_num)]
    
    if slice_runs.empty:
        return f"No data available for Slice {slice_num}."

    # Isolate runs by their role tags
    incumbent = slice_runs[slice_runs["tags.run_role"] == "champion_evaluation"]
    promoted_chal = slice_runs[slice_runs["tags.run_role"] == "promoted_champion"]
    rejected_chal = slice_runs[slice_runs["tags.run_role"] == "rejected_challenger"]
    
    # Build the Markdown Table Header
    md_table = f"### Slice {slice_num} MLOps Audit Report\n\n"
    md_table += "| **Model Role** | **AUC** | **Brier Score** | **Run ID (Audit)** | **Promoted** |\n"
    md_table += "|---|---|---|---|---|\n"
    
    def get_metric(df: pd.DataFrame, metric_name: str) -> str:
        col_name = f'metrics.{metric_name}'
        if not df.empty and col_name in df.columns and pd.notna(df.iloc[0][col_name]):
            return f"{df.iloc[0][col_name]:.4f}"
        return "N/A"
    
    def get_run_id(df: pd.DataFrame) -> str:
        return f"`{df.iloc[0]['run_id'][:8]}`" if not df.empty else "N/A"

    # Row 1: The Incumbent / Baseline Evaluation
    if not incumbent.empty:
        md_table += f"| Incumbent (Champion Eval) | {get_metric(incumbent, 'auc')} | {get_metric(incumbent, 'brier_score')} | {get_run_id(incumbent)} | — |\n"
        
    # Row 2: The Challenger (If one was triggered and evaluated)
    if not promoted_chal.empty:
        md_table += f"| Challenger (Adaptive) | **{get_metric(promoted_chal, 'auc')}** | {get_metric(promoted_chal, 'brier_score')} | {get_run_id(promoted_chal)} | **Yes** |\n"
    elif not rejected_chal.empty:
        md_table += f"| Challenger (Rejected) | {get_metric(rejected_chal, 'auc')} | {get_metric(rejected_chal, 'brier_score')} | {get_run_id(rejected_chal)} | No |\n"
        
    return md_table

print("Generating Audit Reports for Dissertation...\n")
# Assuming EXPERIMENT_NAME is defined in Cell 2. If not, replace with "retail_mlops_champion_challenger"
for i in range(2, 6):
    report_md = generate_slice_report(i, EXPERIMENT_NAME)
    display(Markdown(report_md))

In [ ]:
# Cell 8: Launch MLflow UI
import subprocess
import time

## Method 1 ----------using google collab's output function
# 1. Terminate any existing MLflow UI processes to prevent port conflicts
!pkill -f mlflow

# 2. Start the MLflow UI in the background
print("Starting MLflow tracking server...")
process = subprocess.Popen([
    "mlflow", "ui", 
    "--backend-store-uri", "sqlite:///mlflow.db", 
    "--port", "5000", "allowed-hosts", "*"
])

# Give the server a few seconds to fully boot up
time.sleep(15)

# 3. Expose the UI based on the environment
try:
    # If running inside Google Colab:
    from google.colab import output
    print("Detected Google Colab. Embedding MLflow UI below:")
    #output.serve_kernel_port_as_iframe(5000, width='100%', height=800)
    output.serve_kernel_port_as_window(5000)
    
except ImportError:
    # If running in local Jupyter, VS Code, or another environment:
    print("\n" + "="*50)
    print("🚀 MLflow UI is live!")
    print("👉 Open this link in your browser: http://localhost:5000")
    print("="*50 + "\n")